In [ ]:
# ===== Brain Tumor Segmentation: runner notebook =====
#
# Thin driver around the repository: clones (or updates) it into the working directory, then calls
# the repo's own train.py / test.py. No pipeline code lives here, so the notebook never drifts from
# the repo; behaviour changes go through config.yaml or the RUN_OVERRIDES dict below.
# Hosted runtimes need internet access enabled for the clone.
#
# ===== 1. Get the code =====
# Inside a local checkout (train.py next to the notebook) the checkout is used as-is.
# Otherwise the repo is cloned once and fast-forwarded on re-runs, so a restarted session picks up pushes.

import os, sys, glob, subprocess, importlib

REPO_URL = 'https://github.com/SiL3nTL00p/Brain_Tumor_Segmentation_BCP.git'
BRANCH = 'main'

ON_KAGGLE = os.path.isdir('/kaggle/input')
BASE_DIR = '/kaggle/working' if ON_KAGGLE else os.getcwd()

if os.path.exists(os.path.join(os.getcwd(), 'train.py')):
    REPO_DIR = os.getcwd()
else:
    REPO_DIR = os.path.join(BASE_DIR, 'Brain_Tumor_Segmentation_BCP')
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)

# train.py / test.py run as subprocesses from REPO_DIR; sys.path lets this kernel import the same modules.
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install only missing packages: hosted images pin torch/numpy builds that must not be replaced.
for module, pip_name in [('nibabel', 'nibabel'), ('cv2', 'opencv-python-headless'), ('yaml', 'pyyaml')]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

import torch
print('repo:', REPO_DIR, '| commit:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
      capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| mps', torch.backends.mps.is_available())

In [ ]:
# ===== 2. Data discovery =====
# DATA_ROOT = first directory (depth <= 4) holding BraTS2021_* patient folders with a _seg file.
# Datasets that ship only as .tar are extracted once to a writable temp disk (input mounts are
# read-only); the extracted copy is detected and reused on re-runs. A non-None DATA_ROOT skips the search.

import tarfile

def find_patient_root(base, max_depth=4):
    base_depth = base.rstrip('/').count('/')
    for root, dirs, _ in os.walk(base):
        if root.count('/') - base_depth >= max_depth:
            dirs[:] = []
            continue
        if any(d.startswith('BraTS2021_') and glob.glob(os.path.join(root, d, '*_seg.nii*')) for d in dirs):
            return root
    return None

DATA_ROOT = None   # explicit override; None = auto-discover

if DATA_ROOT is None and ON_KAGGLE:
    DATA_ROOT = find_patient_root('/kaggle/input')
    if DATA_ROOT is None:
        tars = sorted(glob.glob('/kaggle/input/**/*.tar', recursive=True), key=os.path.getsize)
        if tars:
            dest = '/kaggle/temp/brats' if os.path.isdir('/kaggle/temp') else '/tmp/brats'
            os.makedirs(dest, exist_ok=True)
            if find_patient_root(dest) is None:
                print('extracting', tars[-1], '->', dest)
                with tarfile.open(tars[-1]) as t:
                    t.extractall(dest)
            DATA_ROOT = find_patient_root(dest)
elif DATA_ROOT is None:
    DATA_ROOT = find_patient_root(REPO_DIR)

assert DATA_ROOT, 'no BraTS2021_* patient folders found; attach the dataset or set DATA_ROOT'
n_patients = len([d for d in os.listdir(DATA_ROOT) if d.startswith('BraTS2021_')])
print(f'DATA_ROOT = {DATA_ROOT} ({n_patients} patients)')

In [ ]:
# ===== 3. Run config =====
# The repo's config.yaml stays the baseline; RUN_OVERRIDES is deep-merged on top and the result is
# written to config_run.yaml, which every later cell passes to train.py / test.py.
# OUTPUT_DIR sits outside REPO_DIR so re-cloning or pulling never touches finished runs.
# Worker counts track the CPU count (hosted GPU runtimes typically expose 4 cores).

import yaml

OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
CPUS = os.cpu_count() or 2

RUN_OVERRIDES = {
    'paths': {'data_root': DATA_ROOT, 'output_dir': OUTPUT_DIR},
    'data': {'scan_workers': min(4, CPUS)},
    'training': {'num_workers': min(4, CPUS)},
}

def deep_merge(base, extra):
    for k, v in extra.items():
        base[k] = deep_merge(base.get(k, {}), v) if isinstance(v, dict) else v
    return base

def write_config(path, extra=None):
    cfg = yaml.safe_load(open(os.path.join(REPO_DIR, 'config.yaml')))
    deep_merge(cfg, RUN_OVERRIDES)
    deep_merge(cfg, extra or {})
    yaml.safe_dump(cfg, open(path, 'w'), sort_keys=False)
    return path

RUN_CONFIG = write_config(os.path.join(BASE_DIR, 'config_run.yaml'))
print(open(RUN_CONFIG).read())

In [ ]:
# ===== 4. Shape test =====
# U-Net must map (2, 4, 192, 192) -> (2, 1, 192, 192); model.py asserts this when run as a script.
!{sys.executable} -u model.py

In [ ]:
# ===== 5. Smoke test =====
# One epoch on <= 3 patients through the full training path, then asserts that every artifact the
# real run produces exists. Separate output dir keeps it out of the real run history.

SMOKE_DIR = os.path.join(BASE_DIR, 'smoke_outputs')
smoke_cfg = write_config(os.path.join(BASE_DIR, 'config_smoke.yaml'), {
    'paths': {'output_dir': SMOKE_DIR},
    'data': {'max_patients': 3},
    'training': {'epochs': 1},
    'evaluation': {'num_overlay_patients': 1},
    'sanity': {'overfit_one_batch': False},
})
!{sys.executable} -u train.py --config {smoke_cfg} --debug

smoke_run = sorted(glob.glob(os.path.join(SMOKE_DIR, 'run_*')))[-1]
expected = ['results.txt', 'config_used.yaml', 'history.csv', 'checkpoints/best.pt', 'checkpoints/last.pt',
            'graphs/loss_curve.png', 'graphs/dice_iou_curve.png', 'graphs/per_patient_dice.png',
            'graphs/slice_buckets.png', 'predictions/*.nii.gz', 'graphs/overlays/*.png']
missing = [f for f in expected if not glob.glob(os.path.join(smoke_run, f))]
assert not missing, f'smoke test failed, missing: {missing}'
print('smoke test passed:', smoke_run)

In [ ]:
# ===== 6. Overfit-one-batch check =====
# Trains repeatedly on one tumor-containing batch with augmentation off. Loss should approach 0
# (results.txt reports PASS below 0.1). Failure indicates label, loss or shape bugs, not capacity:
# memorising one batch is trivial for a 7.8M-parameter U-Net.

overfit_cfg = write_config(os.path.join(BASE_DIR, 'config_overfit.yaml'), {
    'paths': {'output_dir': os.path.join(BASE_DIR, 'overfit_outputs')},
    'sanity': {'overfit_one_batch': True},
})
!{sys.executable} -u train.py --config {overfit_cfg}

In [ ]:
# ===== 7. Training =====
# Writes OUTPUT_DIR/run_YYYYMMDD_HHMMSS/. best.pt is chosen by mean per-patient 3D Dice on the
# validation split; last.pt is overwritten every epoch so an interrupted run can still be evaluated.
!{sys.executable} -u train.py --config {RUN_CONFIG}

In [ ]:
# ===== 8. Results of the most recent run =====
from IPython.display import Image, display as show
import pandas as pd

RUN_DIR = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'run_*')))[-1]
print(open(os.path.join(RUN_DIR, 'results.txt')).read())
show(pd.read_csv(os.path.join(RUN_DIR, 'history.csv')))
for g in ['slice_buckets', 'loss_curve', 'dice_iou_curve', 'per_patient_dice']:
    show(Image(os.path.join(RUN_DIR, 'graphs', g + '.png')))

In [ ]:
# Panels: FLAIR | ground truth (green) | prediction (red); per patient one tumor-max slice and one healthy slice.
for p in sorted(glob.glob(os.path.join(RUN_DIR, 'graphs', 'overlays', '*.png'))):
    print(os.path.basename(p))
    show(Image(p))

In [ ]:
# ===== 9. Evaluation =====
# Rebuilds the validation split from seed + val_fraction + max_patients, so it matches training only
# while those are unchanged. Writes a separate OUTPUT_DIR/test_* folder.
!{sys.executable} -u test.py --config {RUN_CONFIG} --checkpoint {RUN_DIR}/checkpoints/best.pt

In [ ]:
# ===== 10. Single-patient inference =====
# No _seg file required. Output is a uint8 NIfTI carrying the source scan's affine, so it overlays
# on the original volume in any NIfTI viewer; Dice/IoU are printed only when a _seg file exists.
PATIENT = sorted(glob.glob(os.path.join(DATA_ROOT, 'BraTS2021_*')))[0]
!{sys.executable} -u test.py --config {RUN_CONFIG} --checkpoint {RUN_DIR}/checkpoints/best.pt --patient {PATIENT}

In [ ]:
# ===== 11. Export =====
# Zips OUTPUT_DIR (all runs, checkpoints, predictions, graphs) into BASE_DIR for download.
import shutil
zip_path = shutil.make_archive(os.path.join(BASE_DIR, 'btsp_outputs'), 'zip', OUTPUT_DIR)
print(zip_path, f'{os.path.getsize(zip_path) / 1e6:.1f} MB')